# 🏗️ Notebook 1: Notification System — Requirements & Architecture

## 🛠️ Setup

```bash
cd 06-system-designs/notification-system
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## What we're designing

A service that sends **push / email / SMS** notifications — reliably, at high volume, with
priority control.

### Functional requirements
- A service calls `send(user_id, template, channel)`.
- We deliver via **APNs/FCM** (push), **SMTP** (email), **Twilio** (SMS).
- Users can set preferences (channels they opt out of, quiet hours).
- **Retries** on provider failure.
- **Idempotency**: duplicate sends should not double-notify.

### Non-functional
- **10k/s sustained, 100k/s spikes** (e.g., breaking news).
- **Priority**: a 2FA code must beat a marketing blast.
- **Durability**: don't drop on service restart.


## High-level architecture

```
   callers (other services)
        │ POST /notify
        ▼
  ┌─────────────┐
  │ API gateway │
  └──────┬──────┘
         │
         ▼
  ┌─────────────┐     ┌─────────────┐
  │ Preferences │────▶│ drop if     │
  │ check       │     │ opted-out   │
  └──────┬──────┘     └─────────────┘
         │
         ▼
  ┌────────────────────────────┐
  │ Priority queues (Kafka /   │
  │   RabbitMQ / Redis streams)│
  │  ├─ high   (2FA, security) │
  │  ├─ normal (txn receipts)  │
  │  └─ low    (marketing)     │
  └──────┬─────────────────────┘
         │
         ▼
  ┌────────────────────────┐
  │ Dispatcher workers     │
  │ (per-channel pool)     │
  └──┬───────┬───────┬─────┘
     │       │       │
     ▼       ▼       ▼
    APNs   SMTP   Twilio
```

Each caller picks a priority; separate queues mean low-priority can never starve high-priority
workers. Workers of different sizes drain each queue.
